# CardioRisk AI : prédiction du risque cardiovasculaire

Projet Data Mining & Machine Learning sur le dataset *Heart Disease* (étude de Framingham).

**Organisation :** exploration → preprocessing → data mining non supervisé (ACP, AFC, K-means) → machine learning supervisé (arbre de décision, réseau de neurones) → sauvegarde du modèle utilisé par l'application Streamlit (`app.py`).

> **Exécution :** lance ce notebook depuis le dossier `notebooks/` (ou la racine du dépôt) avec le fichier `data/heart_disease.csv` en place. Les figures sont enregistrées dans `docs/figures/`.

# I. Imports et chargements
## 1. Import des librairies

In [ ]:
#### Gestion des chemins et de l'environnement
import sys
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))          # pour importer train.py
FIG_DIR = ROOT / "docs" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

#### Manipulation de données
import pandas as pd
import numpy as np

#### Visualisation des résultats
import matplotlib.pyplot as plt
import seaborn as sns

#### Preprocessing (exploration)
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor

#### Réduction de dimensionnalité et clustering
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

#### Évaluation et visualisation de l'arbre
from sklearn.tree import plot_tree
from sklearn.metrics import classification_report

#### Tests statistiques (AFC / Chi-Square)
from scipy.stats import chi2_contingency
import prince

#### Partie Machine Learning (pipeline sans fuite d'information)
import train

## 2. Import de la dataset "Heart Disease"

In [ ]:
DATA_PATH = next((p for p in [ROOT / "data" / "heart_disease.csv", Path("heart_disease.csv")]
                  if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("heart_disease.csv introuvable : place-le dans data/")

df_hd = pd.read_csv(DATA_PATH)

# II. Compréhension des données
## 1. Analyse globale de la structure et la distribution des variables

In [ ]:
print(f"\n Dimensions du dataset : {df_hd.shape[0]} lignes × {df_hd.shape[1]} colonnes")
print("\n--- Types de variables ---")
print(df_hd.dtypes)
print("\n--- Aperçu des 5 premières lignes ---")
print(df_hd.head())
print("\n--- Statistiques descriptives des variables numériques ---")
print(df_hd.describe())
print("\n--- Différentes valeurs des variables catégorielles---")
cat_vars = df_hd.select_dtypes(include=['object', 'string']).columns
for col in cat_vars:
    unique_vals = df_hd[col].unique()
    print(f"Variable '{col}' ({len(unique_vals)} catégories) :")
    print(f"  Values: {unique_vals}")
    print("\n" )

## 2. Identification des valeurs manquantes

In [ ]:
print("\n--- Valeurs manquantes ---")
missing = df_hd.isna().sum()
print(missing[missing > 0])

## 3. Identification des valeurs censurées (> or < or LOD "Limit of detection")

In [ ]:
# ── Détection des valeurs censurées (symboles < ou >) ─────────────────────────
censored_found = False

for col in df_hd.columns:
    if pd.api.types.is_string_dtype(df_hd[col]) or df_hd[col].dtype == object:
        mask = df_hd[col].astype(str).str.contains(r'[<>]', regex=True, na=False)
        n = mask.sum()
        if n > 0:
            censored_found = True
            print(f"\n '{col}' — {n} censored value(s) detected :")
            print(df_hd[col][mask].value_counts().to_string())

if not censored_found:
    print("\n No censored values detected in the dataset.")

## 4. Identification des valeurs aberrantes/extrêmes

Les valeurs aberrantes sont **identifiées (méthode IQR) mais conservées** : en médecine, des valeurs extrêmes (tension très élevée, par exemple) peuvent être cliniquement plausibles.

In [ ]:
num_vars = ['age', 'cigsPerDay', 'totChol', 'sysBP',
            'diaBP', 'BMI', 'heartRate', 'glucose']

fig, axes = plt.subplots(3, 3, figsize=(16, 14))
axes = axes.flatten()

print(f"{'Variable':<15} | {'Nbr':<5} | {'Exemples de valeurs extrêmes':<40}")
print("─" * 70)

for i, var in enumerate(num_vars):
    data = df_hd[var].dropna()

    # 1. Calcul des bornes IQR
    q1, q3 = data.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr

    # 2. Identification des outliers
    is_out = (data < lower) | (data > upper)
    outliers = np.sort(data[is_out].unique())
    n_outliers = int(is_out.sum())

    # 3. Boxplot
    axes[i].boxplot(data, patch_artist=True,
                    boxprops=dict(facecolor='#AED6F1', color='#2E75B6'),
                    medianprops=dict(color='red', linewidth=2),
                    flierprops=dict(marker='o', markerfacecolor='salmon', markersize=4, alpha=0.6))

    axes[i].set_title(f"{var}\n({n_outliers} outliers)", fontsize=10, fontweight='bold')
    axes[i].tick_params(axis='x', which='both', bottom=False, labelbottom=False)

    # 4. Affichage textuel formaté
    if len(outliers) > 0:
        sample = list(outliers[:3]) + list(outliers[-2:]) if len(outliers) > 5 else outliers
        out_str = ", ".join([f"{v:.1f}" for v in sample])
    else:
        out_str = "Aucune"

    print(f"{var:<15} | {n_outliers:<5} | {out_str}")

# Le 9e sous-graphique est vide
axes[-1].axis('off')

plt.suptitle("Analyse des Valeurs Aberrantes (Méthode IQR)", fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.savefig(FIG_DIR / "boxplots_outliers.png", dpi=150, bbox_inches='tight')
plt.show()

## 5. Analyse de la distribution de la variable cible "Heart_ stroke"

In [ ]:
print(df_hd['Heart_ stroke'].value_counts())
print("\n")
print(df_hd['Heart_ stroke'].value_counts(normalize=True).mul(100).round(1).astype(str) + " %")

colors = ['#5DADE2', '#E74C3C']

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ── 1. Bar chart
counts = df_hd['Heart_ stroke'].value_counts()
bars = axes[0].bar(counts.index.astype(str), counts.values, color=colors,
                   edgecolor='white', linewidth=2, width=0.6)

for bar in bars:
    height = bar.get_height()
    axes[0].annotate(f'{int(height)}',
                    xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 3), textcoords="offset points",
                    ha='center', va='bottom', fontweight='bold', color='#2C3E50')

axes[0].set_title("Répartition des patients", fontsize=12, pad=15)
axes[0].set_ylabel("Nombre de patients", labelpad=10)
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)
axes[0].grid(axis='y', linestyle='--', alpha=0.3)

# ── 2. Donut chart
wedges, texts, autotexts = axes[1].pie(counts,
                                       labels=counts.index.astype(str),
                                       autopct='%1.1f%%',
                                       startangle=140,
                                       colors=colors,
                                       pctdistance=0.85,
                                       explode=[0, 0.1],
                                       textprops={'fontweight': 'bold'})

centre_circle = plt.Circle((0, 0), 0.60, fc='white')
axes[1].add_artist(centre_circle)
axes[1].set_title("Proportion du risque (%)", fontsize=12, pad=15)

plt.suptitle("Analyse de la variable cible : Risque de maladie cardiaque",
             fontsize=15, fontweight='bold', y=1.05, color='#2C3E50')

plt.tight_layout()
plt.savefig(FIG_DIR / "fig1_target_distribution_pro.png", dpi=200, bbox_inches='tight')
plt.show()

# III. Préparation des données (preprocessing)
## 1. Encodage des variables catégorielles

In [ ]:
df_clean = df_hd.copy()
le = LabelEncoder()
df_clean['Gender_enc'] = le.fit_transform(df_clean['Gender'])       # Male=1, Female=0
df_clean['prevalentStroke_enc'] = le.fit_transform(df_clean['prevalentStroke'])
df_clean['Heart_stroke_enc'] = le.fit_transform(df_clean['Heart_ stroke'])
edu_map = {'uneducated': 0, 'primaryschool': 1, 'graduate': 2, 'postgraduate': 3}
df_clean['education_enc'] = df_clean['education'].map(edu_map)

# Vérification pour le genre
print("Vérification Gender:")
print(pd.crosstab(df_clean['Gender'], df_clean['Gender_enc']))

# Vérification pour l'éducation (mapping manuel)
print("\nVérification Education:")
print(pd.crosstab(df_clean['education'], df_clean['education_enc']))

## 2. Imputation des valeurs manquantes (exploration non supervisée)

Cette imputation est appliquée à **l'ensemble du dataset** : elle ne sert que pour l'exploration non supervisée (ACP, AFC, K-means), où il n'y a pas de découpage train/test. Elle n'utilise pas la cible.

Pour la partie **Machine Learning** (section V), l'imputation est refaite **à l'intérieur du pipeline**, uniquement sur les données d'entraînement, afin d'éviter toute fuite d'information vers le jeu de test.

In [ ]:
# Imputer par Random Forest
rf_imputer = IterativeImputer(
    estimator=RandomForestRegressor(n_estimators=10, random_state=42, n_jobs=-1),
    max_iter=10,
    random_state=42
)

# On retire les colonnes texte (leurs versions encodées sont déjà numériques)
# ainsi que la cible : elle ne doit pas servir à imputer les variables explicatives.
columns_to_drop_before_imputation = ['Gender', 'education', 'prevalentStroke',
                                     'Heart_ stroke', 'Heart_stroke_enc']
df_numerical_for_imputation = df_clean.drop(columns=columns_to_drop_before_imputation, errors='ignore')

print("Imputation Random Forest en cours...")
df_imputed_array = rf_imputer.fit_transform(df_numerical_for_imputation)

df_final = pd.DataFrame(df_imputed_array, columns=df_numerical_for_imputation.columns)
df_final['Heart_stroke_enc'] = df_clean['Heart_stroke_enc'].values

# Post-traitement : arrondir les variables qui doivent être discrètes
cols_to_round = ['Gender_enc', 'prevalentStroke_enc', 'education_enc', 'BPMeds',
                 'cigsPerDay', 'diabetes', 'Heart_stroke_enc']

for col in cols_to_round:
    if col in df_final.columns:
        df_final[col] = df_final[col].round().astype(int)

print("\nImputation terminée avec succès. \n")
print(f"Valeurs manquantes restantes : {df_final.isnull().sum().sum()}")

# Vérification de l'imputation sur un patient
nan_indices = df_clean[df_final.columns].isnull().any(axis=1)
idx_to_check = df_clean[nan_indices].index[0]

avant = df_clean.loc[idx_to_check, df_final.columns]
apres = df_final.loc[idx_to_check]

comparaison = pd.DataFrame({'Avant Imputation': avant, 'Après Imputation': apres})

print("\n")
print("─" * 45)
print(comparaison[comparaison['Avant Imputation'].isnull()])
print("─" * 45)
print("Contexte complet du patient :")
print(apres)

## 3. Normalisation

In [ ]:
features_all = ['age', 'Gender_enc', 'education_enc', 'currentSmoker', 'cigsPerDay',
                'BPMeds', 'prevalentHyp', 'diabetes', 'totChol', 'sysBP',
                'diaBP', 'BMI', 'heartRate', 'glucose']

X = df_final[features_all]
y = df_final['Heart_stroke_enc']      # utilisée pour colorer les graphiques

# --- Normalisation ---
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# IV. Data Mining
## 1. Analyse en composantes principales (ACP)

In [ ]:
# ── ACP sur variables continues uniquement ────────────────────────────────────
num_var = ['age', 'cigsPerDay', 'totChol', 'sysBP', 'diaBP', 'BMI', 'heartRate', 'glucose']

X_num = df_final[num_var].copy()
X_num_scaled = StandardScaler().fit_transform(X_num)

pca = PCA()
pca.fit(X_num_scaled)

explained = pca.explained_variance_ratio_ * 100
cumulative = np.cumsum(explained)
n_comp_80 = np.argmax(cumulative >= 80) + 1

print(f"\nVariance expliquée par composante :")
for i, (e, c) in enumerate(zip(explained, cumulative)):
    print(f"   CP{i+1}: {e:.2f}% (cumulée: {c:.2f}%)")
print(f"\n{n_comp_80} composantes expliquent au moins 80% de la variance")

# ── Scree plot + variance cumulée ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(range(1, len(explained)+1), explained, color='steelblue', edgecolor='black', alpha=0.8)
axes[0].plot(range(1, len(explained)+1), explained, 'ro-', markersize=6)
axes[0].set_xlabel("Composante Principale")
axes[0].set_ylabel("Variance expliquée (%)")
axes[0].set_title("Scree Plot - Eboulis des valeurs propres")
axes[0].axhline(y=5, color='gray', linestyle='--', alpha=0.7, label='Seuil 5%')
axes[0].legend()

axes[1].plot(range(1, len(cumulative)+1), cumulative, 'b-o', markersize=6)
axes[1].axhline(y=80, color='red', linestyle='--', label='80% de variance')
axes[1].axvline(x=n_comp_80, color='green', linestyle='--', label=f'{n_comp_80} composantes')
axes[1].fill_between(range(1, len(cumulative)+1), cumulative, alpha=0.2)
axes[1].set_xlabel("Nombre de composantes")
axes[1].set_ylabel("Variance cumulée (%)")
axes[1].set_title("Variance Cumulée Expliquée")
axes[1].legend()
axes[1].set_ylim([0, 105])
plt.suptitle("ACP - Analyse en Composantes Principales", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / "fig3_pca_scree.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Projection sur CP1-CP2 ────────────────────────────────────────────────────
pca_2 = PCA(n_components=2)
X_pca2 = pca_2.fit_transform(X_num_scaled)

plt.figure(figsize=(10, 7))
colors = {0: 'steelblue', 1: 'salmon'}
labels_text = {0: 'Pas de maladie (No)', 1: 'Maladie cardiaque (Yes)'}
for cls in [0, 1]:
    mask = (y == cls).values
    plt.scatter(X_pca2[mask, 0], X_pca2[mask, 1],
                c=colors[cls], label=labels_text[cls], alpha=0.5, s=20)
plt.xlabel(f"CP1 ({explained[0]:.1f}% de variance)")
plt.ylabel(f"CP2 ({explained[1]:.1f}% de variance)")
plt.title("ACP — Projection des individus sur le plan factoriel (CP1 × CP2)", fontweight='bold')
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "fig4_pca_biplot.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Cercle des corrélations ───────────────────────────────────────────────────
pca_full = PCA(n_components=2)
pca_full.fit(X_num_scaled)
loadings = pca_full.components_.T             # shape (8, 2)

plt.figure(figsize=(9, 9))
ax = plt.gca()
circle = plt.Circle((0, 0), 1, fill=False, color='gray', linestyle='--')
ax.add_patch(circle)
for i, feature in enumerate(num_var):
    ax.arrow(0, 0, loadings[i, 0], loadings[i, 1],
             head_width=0.03, head_length=0.03, fc='steelblue', ec='steelblue')
    ax.text(loadings[i, 0]*1.12, loadings[i, 1]*1.12, feature,
            ha='center', va='center', fontsize=9, color='darkblue')
ax.axhline(0, color='k', linewidth=0.5)
ax.axvline(0, color='k', linewidth=0.5)
ax.set_xlim(-1.3, 1.3)
ax.set_ylim(-1.3, 1.3)
ax.set_xlabel(f"CP1 ({explained[0]:.1f}%)")
ax.set_ylabel(f"CP2 ({explained[1]:.1f}%)")
ax.set_title("Cercle des Corrélations (ACP)", fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / "fig5_pca_correlation_circle.png", dpi=150, bbox_inches='tight')
plt.show()

### Cos² et contributions des variables

In [ ]:
# ── ACP : Cos2 et Contributions ──────────────────────────────────────────────
pca = PCA()
pca.fit(X_num_scaled)
components = pca.components_
explained = pca.explained_variance_ratio_ * 100
eigenvalues = pca.explained_variance_

cos2 = components.T ** 2
contrib = (components.T ** 2 / eigenvalues) * 100

cos2_df = pd.DataFrame(cos2, index=num_var,
                       columns=[f"CP{i+1}" for i in range(len(explained))])
contrib_df = pd.DataFrame(contrib, index=num_var,
                          columns=[f"CP{i+1}" for i in range(len(explained))])

seuil = 100 / len(num_var)

print("─" * 55)
print("COS2 — Qualité de représentation des variables (CP1-CP5)")
print("─" * 55)
print(cos2_df.iloc[:, :5].round(3).to_string())

print("\n" + "─" * 55)
print("CONTRIBUTIONS (%) des variables aux composantes (CP1-CP5)")
print("─" * 55)
print(contrib_df.iloc[:, :5].round(2).to_string())

# ── Figure : 4 graphiques ─────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(18, 14))

sns.heatmap(cos2_df.iloc[:, :5], annot=True, fmt=".2f", cmap="YlOrRd",
            linewidths=0.5, ax=axes[0, 0], vmin=0, vmax=1, annot_kws={"size": 9})
axes[0, 0].set_title("Cos2 — Qualité de représentation\n(CP1 à CP5)", fontweight='bold', fontsize=12)
axes[0, 0].set_xlabel("Composantes Principales")
axes[0, 0].set_ylabel("Variables")

sns.heatmap(contrib_df.iloc[:, :5], annot=True, fmt=".1f", cmap="Blues",
            linewidths=0.5, ax=axes[0, 1], annot_kws={"size": 9})
axes[0, 1].set_title(f"Contributions (%) aux composantes\n(seuil uniforme = {seuil:.1f}%)",
                     fontweight='bold', fontsize=12)
axes[0, 1].set_xlabel("Composantes Principales")
axes[0, 1].set_ylabel("Variables")

cos2_plan = cos2_df["CP1"] + cos2_df["CP2"]
cos2_plan_sorted = cos2_plan.sort_values(ascending=True)
colors_cos2 = ['#E74C3C' if v < 0.4 else '#2ECC71' for v in cos2_plan_sorted]
axes[1, 0].barh(cos2_plan_sorted.index, cos2_plan_sorted.values,
                color=colors_cos2, edgecolor='black')
axes[1, 0].axvline(x=0.4, color='red', linestyle='--', linewidth=1.5, label='Seuil cos2 = 0.4')
axes[1, 0].set_title("Cos2 cumulé CP1+CP2\n(vert = bien représenté, rouge = mal représenté)",
                     fontweight='bold', fontsize=12)
axes[1, 0].set_xlabel("Cos2 (CP1 + CP2)")
axes[1, 0].legend()

contrib_cp1_sorted = contrib_df["CP1"].sort_values(ascending=True)
colors_contrib = ['#E74C3C' if v < seuil else '#2196F3' for v in contrib_cp1_sorted]
axes[1, 1].barh(contrib_cp1_sorted.index, contrib_cp1_sorted.values,
                color=colors_contrib, edgecolor='black')
axes[1, 1].axvline(x=seuil, color='red', linestyle='--', linewidth=1.5,
                   label=f'Seuil uniforme = {seuil:.1f}%')
axes[1, 1].set_title(f"Contributions à CP1\n(bleu = au-dessus du seuil uniforme {seuil:.1f}%)",
                     fontweight='bold', fontsize=12)
axes[1, 1].set_xlabel("Contribution (%)")
axes[1, 1].legend()

plt.suptitle("ACP — Cos2 et Contributions des Variables", fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(FIG_DIR / "pca_cos2_contrib.png", dpi=150, bbox_inches='tight')
plt.show()

## 2. Analyse factorielle des correspondances (AFC)

Le tabagisme (`cigsPerDay` discrétisé en 4 classes) est croisé avec trois variables binaires. Chaque croisement est testé par un Chi², puis représenté par les coordonnées de l'AFC et les profils lignes / colonnes.

In [ ]:
# ── Discrétisation de cigsPerDay ──────────────────────────────────────────────
df_final['cigs_cat'] = pd.cut(df_final['cigsPerDay'],
                              bins=[-1, 0, 10, 20, 100],
                              labels=['Non-fumeur', 'Léger (1-10)',
                                      'Modéré (11-20)', 'Intense (>20)'])

afc_pairs = [
    ('cigs_cat', 'prevalentStroke_enc', 'Tabagisme × AVC préalable',
     {0: 'No Stroke', 1: 'Stroke'}),
    ('cigs_cat', 'prevalentHyp', 'Tabagisme × Hypertension',
     {0: 'No Hyp', 1: 'Hypertension'}),
    ('cigs_cat', 'Gender_enc', 'Tabagisme × Genre',
     {0: 'Female', 1: 'Male'}),
]

for var1, var2, title, label_map in afc_pairs:

    df_final[f'{var2}_label'] = df_final[var2].map(label_map)
    ct = pd.crosstab(df_final['cigs_cat'], df_final[f'{var2}_label'])

    print("=" * 55)
    print(f"AFC : {title}")
    print("=" * 55)
    print(ct)

    chi2, p, dof, _ = chi2_contingency(ct)
    print(f"\nChi² = {chi2:.3f} | p-value = {p:.4f} | ddl = {dof}")
    print("\n Association SIGNIFICATIVE" if p < 0.05 else "Pas d'association significative \n")

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # ── 1. AFC : biplot 2D ou coordonnées sur CP1 selon le nombre de composantes ──
    n_components = min(ct.shape[0] - 1, ct.shape[1] - 1)
    ca = prince.CA(n_components=n_components, random_state=42)
    ca = ca.fit(ct)
    row_coords = ca.row_coordinates(ct)
    col_coords = ca.column_coordinates(ct)

    if n_components >= 2:
        # Biplot 2D tracé avec matplotlib (indépendant de la version de prince)
        axes[0].scatter(row_coords.iloc[:, 0], row_coords.iloc[:, 1],
                        c='#2E75B6', s=80, label='Intensité tabagique')
        axes[0].scatter(col_coords.iloc[:, 0], col_coords.iloc[:, 1],
                        c='#C0392B', marker='^', s=80, label=var2)
        for name, r in row_coords.iterrows():
            axes[0].annotate(str(name), (r.iloc[0], r.iloc[1]))
        for name, r in col_coords.iterrows():
            axes[0].annotate(str(name), (r.iloc[0], r.iloc[1]))
        axes[0].axhline(0, color='black', linewidth=0.5)
        axes[0].axvline(0, color='black', linewidth=0.5)
        axes[0].legend()
        axes[0].set_title(f"Biplot AFC\n(Chi²={chi2:.1f}, p={p:.4f})", fontweight='bold')
    else:
        # Une seule composante → barplot des coordonnées sur CP1
        labels_cp1 = [str(i) for i in row_coords.index] + [str(i) for i in col_coords.index]
        values_cp1 = np.concatenate([row_coords.iloc[:, 0].values, col_coords.iloc[:, 0].values])
        colors_cp1 = ['#2E75B6'] * len(row_coords) + ['#C0392B'] * len(col_coords)

        axes[0].barh(labels_cp1, values_cp1, color=colors_cp1, edgecolor='black')
        axes[0].axvline(x=0, color='black', linewidth=1)
        axes[0].set_title(f"Coordonnées sur CP1\n(Chi²={chi2:.1f}, p={p:.4f})\n"
                          f"bleu=lignes, rouge=colonnes", fontweight='bold')
        axes[0].set_xlabel("Coordonnée CP1")

    # ── 2. Profils lignes ─────────────────────────────────────────────────────
    ct_row = ct.div(ct.sum(axis=1), axis=0) * 100
    ct_row.plot(kind='bar', ax=axes[1], edgecolor='black', color=['steelblue', 'salmon'])
    axes[1].set_title(f"Profils lignes\n(% {var2} par intensité tabagique)", fontweight='bold')
    axes[1].set_ylabel("% de patients")
    axes[1].tick_params(axis='x', rotation=25)
    axes[1].legend(title=var2)
    axes[1].axhline(y=50, color='gray', linestyle='--', alpha=0.5)

    # ── 3. Profils colonnes (transposé : une barre par modalité de var2) ─────
    ct_col = ct.div(ct.sum(axis=0), axis=1) * 100
    ct_col.T.plot(kind='bar', ax=axes[2], edgecolor='black',
                  color=['#2ecc71', '#e67e22', '#9b59b6', '#1abc9c'])
    axes[2].set_title(f"Profils colonnes\n(% intensité tabagique par {var2})", fontweight='bold')
    axes[2].set_ylabel("% de patients")
    axes[2].tick_params(axis='x', rotation=0)
    axes[2].legend(title="Intensité tabagique")

    plt.suptitle(f"AFC — {title}", fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"fig_afc_{var2}.png", dpi=150, bbox_inches='tight')
    plt.show()

## 3. K-means

In [ ]:
# ── K-Means sur les 8 variables continues (cohérent avec l'ACP) ───────────────
# ── Méthode du coude ──────────────────────────────────────────────────────────
inertias = []
K_range = range(2, 11)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_num_scaled)
    inertias.append(km.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(K_range, inertias, 'bo-', markersize=8, linewidth=2)
plt.axvline(x=3, color='red', linestyle='--', label='k=3 (coude)')
plt.xlabel("Nombre de clusters (k)")
plt.ylabel("Inertie intra-cluster")
plt.title("Méthode du Coude — Choix du k optimal", fontweight='bold')
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "fig6_kmeans_elbow.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Application K-Means k=3 ───────────────────────────────────────────────────
k_opt = 3
kmeans = KMeans(n_clusters=k_opt, random_state=42, n_init=10)
df_final['cluster'] = kmeans.fit_predict(X_num_scaled)

print(f"\nK-Means avec k={k_opt} :")
print(f"   Inertie totale : {kmeans.inertia_:.2f}")
print("\n--- Profil moyen par cluster ---")
cluster_profile = df_final.groupby('cluster')[num_var].mean().round(3)
print(cluster_profile)

# ── Visualisation sur le plan ACP ─────────────────────────────────────────────
pca_vis = PCA(n_components=2)
X_vis = pca_vis.fit_transform(X_num_scaled)
cluster_colors = ['#2196F3', '#FF5722', '#4CAF50']

plt.figure(figsize=(10, 7))
for c in range(k_opt):
    mask = df_final['cluster'] == c
    size_c = mask.sum()
    plt.scatter(X_vis[mask, 0], X_vis[mask, 1],
                c=cluster_colors[c], label=f"Cluster {c} (n={size_c})",
                alpha=0.5, s=25)
centers_vis = pca_vis.transform(kmeans.cluster_centers_)
plt.scatter(centers_vis[:, 0], centers_vis[:, 1],
            c='black', marker='X', s=200, zorder=10, label='Centroïdes')
plt.xlabel(f"CP1 ({pca_vis.explained_variance_ratio_[0]*100:.1f}%)")
plt.ylabel(f"CP2 ({pca_vis.explained_variance_ratio_[1]*100:.1f}%)")
plt.title(f"K-Means (k={k_opt}) — Projection sur le plan ACP", fontweight='bold')
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "fig7_kmeans_clusters.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Taux de maladie par cluster ───────────────────────────────────────────────
cluster_disease = df_final.groupby('cluster')['Heart_stroke_enc'].mean() * 100

print("\n--- Taux de maladie cardiaque par cluster ---")
for c, val in cluster_disease.items():
    print(f"   Cluster {c} : {val:.1f}%")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
cluster_disease.plot(kind='bar', ax=axes[0], color=cluster_colors, edgecolor='black')
axes[0].set_title("Taux de Maladie Cardiaque par Cluster (%)")
axes[0].set_xlabel("Cluster")
axes[0].set_ylabel("% maladie cardiaque")
axes[0].tick_params(axis='x', rotation=0)

df_final.groupby('cluster').size().plot(kind='bar', ax=axes[1],
                                        color=cluster_colors, edgecolor='black')
axes[1].set_title("Taille de chaque cluster")
axes[1].set_xlabel("Cluster")
axes[1].set_ylabel("Nombre de patients")
axes[1].tick_params(axis='x', rotation=0)

plt.suptitle("Analyse des Clusters K-Means", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / "fig8_kmeans_analysis.png", dpi=150, bbox_inches='tight')
plt.show()

# V. Machine Learning

## Méthode : un pipeline sans fuite d'information

Le code d'entraînement est dans `train.py` (importé ici) pour que le notebook et le script `python train.py` donnent exactement les mêmes résultats. Principes :

1. **Séparation train/test (80/20, stratifiée) avant tout prétraitement.**
2. **Imputation Random Forest + standardisation à l'intérieur du `Pipeline`** : apprises sur l'entraînement seulement, refaites dans chaque pli de validation croisée.
3. **Hyperparamètres de l'arbre choisis par validation croisée à 5 plis** (`GridSearchCV`, critère AUC ROC) sur le jeu d'entraînement ; le jeu de test n'est utilisé qu'une fois, pour l'évaluation finale.
4. **Probabilités calibrées** (`CalibratedClassifierCV`) : l'arbre est entraîné avec `class_weight='balanced'`, ce qui gonfle ses probabilités ; la calibration les rend comparables à de vraies fréquences.
5. **Métriques adaptées à une cible déséquilibrée** : rappel, F1 et AUC ROC en plus de l'accuracy.

> L'imputation étant faite dans le pipeline, les variables binaires imputées ne sont pas arrondies (contrairement à la section III, utilisée pour l'exploration seulement).

In [ ]:
res = train.run_experiment(DATA_PATH)

## 1. Arbre de décision

In [ ]:
# ── Choix de la profondeur par validation croisée ─────────────────────────────
train.plot_depth_curve(res["grid"], FIG_DIR / "fig11_dt_depth_selection.png")
plt.show()
print(f"Meilleurs hyperparamètres : {res['grid'].best_params_}")

# ── Résultats sur le jeu de test ──────────────────────────────────────────────
m = res["metrics"]["Arbre de décision"]
print(f"\nAccuracy test       : {m['accuracy']*100:.2f}%")
print(f"CV Accuracy (5-fold): {m['cv_accuracy']*100:.2f}% ± {m['cv_accuracy_std']*100:.2f}%")
print(f"AUC ROC test        : {m['roc_auc']:.3f}")
print("\nRapport de classification :")
print(classification_report(res["y_test"], res["tree_pred"], target_names=['No (0)', 'Yes (1)']))

In [ ]:
# ── Importance des variables ──────────────────────────────────────────────────
train.plot_importances(res["importances"], FIG_DIR / "fig13_feature_importances.png")
plt.show()

# ── Visualisation de l'arbre (seuils exprimés en variables standardisées) ─────
plt.figure(figsize=(20, 8))
plot_tree(res["tree"].named_steps["clf"],
          feature_names=train.FEATURES,
          class_names=['No', 'Yes'],
          filled=True, rounded=True,
          max_depth=3, fontsize=9,
          impurity=False, proportion=True)
plt.title("Arbre de décision (visualisation limitée à max_depth=3)", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / "fig14_decision_tree_plot.png", dpi=150, bbox_inches='tight')
plt.show()

## 2. Réseau de neurones

In [ ]:
m = res["metrics"]["Réseau de neurones"]
mlp_clf = res["mlp"].named_steps["clf"]
print(f"Accuracy test       : {m['accuracy']*100:.2f}%")
print(f"CV Accuracy (5-fold): {m['cv_accuracy']*100:.2f}% ± {m['cv_accuracy_std']*100:.2f}%")
print(f"AUC ROC test        : {m['roc_auc']:.3f}")
print(f"Itérations          : {mlp_clf.n_iter_}")
print("\nRapport de classification :")
print(classification_report(res["y_test"], res["mlp_pred"], target_names=['No (0)', 'Yes (1)']))

# ── Courbe de convergence ─────────────────────────────────────────────────────
plt.figure(figsize=(8, 5))
plt.plot(mlp_clf.loss_curve_, color='steelblue', linewidth=2, label='Perte (entraînement)')
if mlp_clf.validation_scores_ is not None:
    plt.plot([1 - s for s in mlp_clf.validation_scores_],
             color='salmon', linewidth=2, label='Erreur validation (1 - accuracy)')
plt.xlabel("Itérations")
plt.ylabel("Perte / Erreur")
plt.title("Courbe de convergence du réseau de neurones", fontweight='bold')
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "fig16_mlp_convergence.png", dpi=150, bbox_inches='tight')
plt.show()

## 3. Comparaison des 2 modèles

In [ ]:
table = pd.DataFrame(res["metrics"]).T[["accuracy", "cv_accuracy", "recall", "precision", "f1", "roc_auc"]]
table.columns = ["Accuracy test", "CV accuracy", "Rappel (Yes)", "Précision (Yes)", "F1 (Yes)", "AUC ROC"]
display(table.round(3))

train.plot_confusions(res, FIG_DIR / "fig12_confusion_matrices.png")
plt.show()
train.plot_comparison(res["metrics"], FIG_DIR / "fig15_models_comparison.png")
plt.show()

## 4. Calibration des probabilités

Le modèle déployé est l'arbre dont les probabilités ont été calibrées. Le score de Brier mesure l'écart entre probabilités prédites et fréquences observées (plus bas = mieux).

In [ ]:
from sklearn.calibration import calibration_curve

print(f"Score de Brier : {res['calibration']['brier_avant']:.4f} avant calibration "
      f"→ {res['calibration']['brier_apres']:.4f} après")

plt.figure(figsize=(7, 6))
for name, model, X_eval in [("Arbre (class_weight='balanced')", res["tree"], res["X_test"]),
                            ("Arbre calibré (modèle déployé)", res["tree_cal"], res["X_test_imp"])]:
    proba = model.predict_proba(X_eval)[:, 1]
    frac_pos, mean_pred = calibration_curve(res["y_test"], proba, n_bins=8, strategy="quantile")
    plt.plot(mean_pred, frac_pos, "o-", label=name)
plt.plot([0, 1], [0, 1], "k--", label="Calibration parfaite")
plt.xlabel("Probabilité prédite")
plt.ylabel("Fréquence observée")
plt.title("Courbe de calibration (jeu de test)", fontweight='bold')
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "fig17_calibration.png", dpi=150, bbox_inches='tight')
plt.show()

## 5. Sauvegarde du modèle

Écrit `models/model_heart.pkl` (standardisation + arbre calibré), `models/feature_importances.json` et `models/metrics.json`. L'application `app.py` les charge directement ; aucun `scaler` séparé n'est nécessaire.

L'imputeur Random Forest n'est **pas** sauvegardé : il alourdirait le fichier de plusieurs centaines de Mo, et l'application fournit toujours des valeurs complètes.

In [ ]:
train.save_artifacts(res, ROOT)
size_mb = (ROOT / "models" / "model_heart.pkl").stat().st_size / 1e6
print(f"Modèle et métriques enregistrés dans {ROOT / 'models'} (model_heart.pkl : {size_mb:.2f} Mo)")

## 6. Application Streamlit

L'application est dans `app.py` à la racine du dépôt :

```bash
streamlit run app.py
```